# Actividad 5 — Autocorrelación Espacial Local
## Elecciones presidenciales Chile 2025 — Región Metropolitana

En esta actividad aplicarás los **estadísticos locales** del I de Moran ($I_i$) y del G de Getis-Ord ($G_i^*$) a los resultados de la **segunda vuelta presidencial de 2025** en las comunas de la **Región Metropolitana**.

El objetivo es identificar:

- **Hot spots** (concentraciones de voto Kast) y **cold spots** (concentraciones de voto Jara),
- **Outliers espaciales**: comunas que votan distinto a su entorno,

## Instrucciones

1. Ejecuta primero las celdas de **Setup** (cargan datos y construyen la cartografía).
2. Luego resuelve cada ejercicio en las celdas marcadas con `# Tu código aquí`.
3. Responde las preguntas de interpretación directamente en celdas markdown (cuando aplique).

## Setup — Carga de datos y matriz de pesos

Ejecuta estas celdas **sin modificarlas**.

In [1]:
# Gráficos
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn
import contextily

# Análisis espacial
import geopandas
import pandas
import numpy as np
import esda
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local
from libpysal import weights
from numpy.random import seed

# Visualización específica LISA
import splot
from splot.esda import plot_moran, lisa_cluster, plot_local_autocorrelation

In [4]:
# Cargar resultados electorales de la segunda vuelta 2025 (RM)
elec = pandas.read_csv(BASE / "../../datos/external/elecciones2025/resultados_2v_rm.csv")

# Cargar geometrías de comunas de la RM (Censo 2017)
comunas = geopandas.read_file(BASE / "censo2017/R13/COMUNA_C17.shp")

# Unir datos electorales con geometrías y reproyectar a Web Mercator
db = comunas.merge(elec, on="NOM_COMUNA", how="inner")
db = db.to_crs(epsg=3857)

print(f"Comunas con datos: {len(db)}")
db[["NOM_COMUNA", "Pct_Kast", "Pct_Jara", "Ganador"]].head()

Comunas con datos: 52


,NOM_COMUNA,Pct_Kast,Pct_Jara,Ganador
0,PAINE,58.9,41.1,Kast
1,BUIN,58.4,41.6,Kast
2,PUDAHUEL,48.6,51.4,Jara
3,CERRO NAVIA,42.3,57.7,Jara
4,COLINA,57.6,42.4,Kast


In [6]:
# Matriz de pesos espaciales (k-vecinos más cercanos, k=5)
w = weights.KNN.from_dataframe(db, k=5)
w.transform = "R"   # estandarización por filas



---
## Ejercicio 1 — I de Moran local sobre Pct_Kast (20 min)

Vamos a calcular el **I de Moran local** ($I_i$) para el porcentaje de voto Kast en cada comuna. Recuerda que cada $I_i$ clasifica la comuna en uno de cuatro cuadrantes del gráfico de Moran:

| Código | Cuadrante | Significado |
|:------:|:---------:|:------------|
| 1 | HH | alto Kast rodeado de alto Kast (*hot spot*) |
| 2 | LH | bajo Kast rodeado de alto Kast (*doughnut*) |
| 3 | LL | bajo Kast rodeado de bajo Kast (*cold spot*) |
| 4 | HL | alto Kast rodeado de bajo Kast (*diamond*) |

**1.1** Calcula el **I de Moran local** para cada comuna

In [7]:
# Tu código aquí

**1.2** Guarda en `db` tres columnas nuevas con los atributos principales del objeto `lisa`:
- `Ii` ← (valor del $I_i$ local)
- `q_cuadrante` ← (cuadrante: 1=HH, 2=LH, 3=LL, 4=HL)
- `p_sim` ←  (pseudo p-valor)

In [ ]:
# Tu código aquí


**1.3** Construye una columna `spots` que codifique la clasificación final: si el área NO es significativa ($p \geq 0.05$), debe valer 0; si es significativa, debe conservar el código de cuadrante.

Luego crea una columna con los spots labels, "No significativo","HH","LH", "LL","HL".


Finalmente imprime el conteo de cada categoría.

In [ ]:
# Tu código aquí


**1.4** Crea un **mapa LISA** considerando un `p=0.05` (umbral de significancia). Agrega un título descriptivo.

In [ ]:
# Tu código aquí


**1.5** Identifica **las comunas HH**  (Kast lovers) y **las comunas LL** (Jara lovers Jara).

In [ ]:
# Tu código aquí


**1.6 — Interpretación.** Responde en una celda markdown:

- ¿Qué comunas forman los son Kast lovers? ¿Se agrupan geográficamente en alguna zona específica de Santiago?
- ¿Qué comunas forman los Jara lovers? ¿En qué sector de la ciudad están?

*(Escribe tu respuesta aquí)*

---
## Ejercicio 2 — Outliers espaciales (15 min)

Los outliers espaciales son comunas que **votan distinto a sus vecinas**. En el I de Moran local aparecen como:

- **LH** (*doughnut*): bajo Kast rodeado de alto Kast — una isla Jara en territorio Kast.
- **HL** (*diamond*): alto Kast rodeado de bajo Kast — una isla Kast en territorio Jara.

**2.1** Lista todas las comunas clasificadas como **LH** y **HL** (outliers). Muestra `NOM_COMUNA`, `Pct_Kast` y el cuadrante.

In [ ]:
# Tu código aquí


---
## Ejercicio 3 — Getis-Ord local $G_i^*$ (20 min)

El $G_i^*$ es el estadístico estándar para *hot spot analysis*. A diferencia del I de Moran local, **no distingue outliers** — solo clasifica en hot / cold / no-significativo.

La clasificación se hace con el **z-score** y el **p-valor**:

| Condición | Clasificación |
|:---:|:---:|
| $z > 0$ y $p < 0.05$ | Hot spot |
| $z < 0$ y $p < 0.05$ | Cold spot |
| $p \geq 0.05$ | No significativo |

**3.1** Calcula el $G_i^*$ local con `G_Local(db["Pct_Kast"], w)`. 

In [ ]:
# Tu código aquí


**3.2** Guarda el **z-score** y el **p-valor** en las columnas `Gistar_z` y `Gistar_p` de `db`.

In [ ]:
# Tu código aquí


**3.3** Clasifica cada comuna como *Hot spot*, *Cold spot* o *No significativo* siguiendo la regla de la tabla. Guarda el resultado en la columna `Gistar_cat`. Imprime el conteo.

In [ ]:
# Tu código aquí


**3.4** Dibuja un **mapa de hot/cold spots** Agrega leyenda y título descriptivo.

In [ ]:
# Tu código aquí


**3.5 — Comparación.** Pon lado a lado el **mapa LISA** del Ejercicio 1 y el **mapa Gi\*** de este ejercicio. ¿Dónde coinciden? ¿Dónde difieren? ¿Qué áreas desaparecen en el mapa Gi\* que sí estaban en el LISA?

*(Escribe tu respuesta aquí)*